# 02 - Model Experiments and Evaluation

Notebook for testing feature sets, training settings, and evaluation outputs.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_preprocessing import load_data, handle_missing_values, encode_gender, remove_outliers_iqr
from src.feature_engineering import engineer_all_features, get_feature_columns_for_model
from src.model_training import train_and_compare, get_feature_importance_tree
from src.evaluation import print_metrics_summary
from src.utils import TARGET_COLUMN

print('Project root:', PROJECT_ROOT)

In [ ]:
df, is_synthetic = load_data(project_root=PROJECT_ROOT)
df = handle_missing_values(df, strategy='drop')
df = encode_gender(df, copy=True)
df = engineer_all_features(df, copy=True, include_leaky=True)

numeric_for_iqr = [c for c in df.select_dtypes(include=[np.number]).columns if c not in ['student_id', TARGET_COLUMN]]
df = remove_outliers_iqr(df, columns=numeric_for_iqr, factor=1.5)
df.shape

In [ ]:
# Baseline feature set (no leakage)
base_features = [c for c in get_feature_columns_for_model(include_final_grade=False, include_grade_gap=False) if c in df.columns]
X_base = df[base_features]
y = df[TARGET_COLUMN]

out_base = train_and_compare(X_base, y, feature_names=base_features, test_size=0.2, random_state=42)
out_base['results_df']

In [ ]:
# Experimental feature set (includes final_grade, for comparison only)
exp_features = [c for c in get_feature_columns_for_model(include_final_grade=True, include_grade_gap=False) if c in df.columns]
X_exp = df[exp_features]

out_exp = train_and_compare(X_exp, y, feature_names=exp_features, test_size=0.2, random_state=42)
out_exp['results_df']

In [ ]:
print('Best baseline model:', out_base['best_model_name'])
print('Best experimental model:', out_exp['best_model_name'])

X_test = out_base['X_test_scaled'] if out_base['preprocessor']['best_uses_scaled'] else out_base['X_test']
pred = out_base['best_model'].predict(X_test)
metrics = print_metrics_summary(out_base['y_test'], pred, model_name='Baseline Best Model')
metrics

In [ ]:
imp = get_feature_importance_tree(out_base['best_model'], out_base['preprocessor']['feature_names'])
if imp is not None:
    imp.head(15)
else:
    print('Best model has no tree-based feature importance (likely linear model).')